# Part 1 Web Scraper
In this notebook you will learn how to build a **polite recursive web crawler** using `requests` and `BeautifulSoup4`.

**What you will learn**
- How to scrape multiple related websites safely
- How a **recursive crawler** (BFS-style) works
- How to respect website rules (rate limiting, domain restriction)
- How to clean HTML into readable text
- How to save data for later use (e.g., RAG systems or chatbots)

**Target websites**  
- Innovation Wing: https://innowings.engg.hku.hk/  
- InnoAcademy: https://innoacademy.engg.hku.hk/

By the end you will have a JSON dataset that can be used to power AI applications!

## 1.1. Introduction to Web Scraping

Web scraping is the process of automatically downloading and extracting information from websites.

**Why do we need a crawler?**  
Instead of scraping one page manually, a **recursive crawler** follows links and discovers new pages automatically — just like a search engine bot.

**Important rules we follow**
- Only scrape pages we are allowed to (domain restriction)
- Be polite: wait between requests (`DELAY`)
- Use a realistic User-Agent so servers know it’s a student project
- Stop after a safe limit (`MAX_PAGES`)

This notebook is designed for the **Chatbot Challenge** — the scraped data will become the knowledge base for an intelligent assistant.

## 1.2. Install and Import Libraries

We use two main libraries:
- **`requests`** → downloads raw HTML from the internet
- **`beautifulsoup4` (bs4)** → parses HTML and lets us extract text and links easily

We also import helper modules for URL handling, JSON saving, and timing.

In [1]:
!pip install bs4 requests

In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import json
import time
import os
from typing import List, Dict
from pathlib import Path
BASE_DIR = Path.cwd().parent

## 1.3. Define Constraints & Configuration

Before we start crawling, we need to set clear **rules** so the crawler stays safe and focused.

### Seed URLs
These are the starting points of our crawl.

### Allowed Domains
We **only** allow links from the two official HKU Innovation Wing domains.  
This prevents the crawler from accidentally wandering into the entire internet!

### Other Safety Settings
- `MAX_PAGES`: stops after 150 pages (safety limit)
- `DELAY`: waits 1 second between requests (polite crawling)
- `HEADERS`: defines the browser type used

What websites to scrape from?

In [4]:
SEED_URLS = [
    "https://innowings.engg.hku.hk/",
    "https://innoacademy.engg.hku.hk/",   
]

Restrict recursive crawler to websites owned by Innowing.

In [5]:
ALLOWED_DOMAINS = {"innowings.engg.hku.hk", "innoacademy.engg.hku.hk"}

# Check if a URL belongs to Innowing
def is_internal_link(url: str) -> bool:
    parsed = urlparse(url)
    return parsed.netloc in ALLOWED_DOMAINS or not parsed.netloc  # allow relative links

Other constraints

In [6]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/133.0 Safari/537.36"
}
MAX_PAGES = 20          # Safety limit - increase if needed
DELAY = 1.0              # Seconds between requests (polite enough for this small site)

## 1.4. Helper Functions

We create three clean, reusable functions:

1. **`is_internal_link()`** – checks whether a link belongs to our allowed domains
2. **`simple_clean_text()`** – removes scripts, styles, and extra whitespace (i.e. the website code that is not related to content)
3. **`scrape_page()`** – downloads one page and returns clean text

**Note**: There was a duplicate `is_internal_link` definition earlier — we only need it once!

In [7]:
# Performs the most basic cleaning: removes only JavaScript and CSS, then extracts readable text while reducing extra blank lines.
def simple_clean_text(soup: BeautifulSoup) -> str:
    # Very basic cleaning - remove script/style only
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    # Remove excessive blank lines
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

# Downloads a single page and returns a dictionary containing the URL and its cleaned text (or an error message)
def scrape_page(url: str) -> Dict[str, str]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        text = simple_clean_text(soup)
        return {"url": url, "text": text}
    except Exception as e:
        print(f"❌ Failed {url}: {e}")
        return {"url": url, "text": f"[ERROR: {e}]"}

## 1.5. How the Recursive Crawler Works

We use a **queue** (Breadth-First Search) to explore pages:

1. Start with the two seed URLs
2. Scrape the current page
3. Extract all links on that page
4. Add any new internal links to the queue
5. Repeat until we reach the maximum number of pages

This is exactly how real web crawlers (like Googlebot) explore the web!

### Perform web crawling

In [8]:
print("🚀 Starting crawling...\n")

visited: List[str] = []
queue: List[str] = SEED_URLS[:]
documents: List[Dict[str, str]] = []

while queue and len(documents) < MAX_PAGES:
    url = queue.pop(0)
    if url in visited:
        continue

    print(f"📄 [{len(documents)+1}/{MAX_PAGES}] Scraping → {url}")
    doc = scrape_page(url)
    
    if doc["text"].strip():
        documents.append(doc)
    
    visited.append(url)

    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")
        for a in soup.find_all("a", href=True):
            full_url = urljoin(url, a["href"])
            if is_internal_link(full_url) and full_url not in visited:
                queue.append(full_url)
    except:
        pass  # ignore link extraction errors

    time.sleep(DELAY)

print(f"\n✅ Crawl finished! Collected {len(documents)} pages.")

🚀 Starting crawling...

📄 [1/20] Scraping → https://innowings.engg.hku.hk/
📄 [2/20] Scraping → https://innoacademy.engg.hku.hk/
📄 [3/20] Scraping → https://innowings.engg.hku.hk/#content
📄 [4/20] Scraping → https://innowings.engg.hku.hk/innowing1/
📄 [5/20] Scraping → https://innowings.engg.hku.hk/innowing-two/
📄 [6/20] Scraping → https://innoacademy.engg.hku.hk/#content
📄 [7/20] Scraping → https://innoacademy.engg.hku.hk/aboutus/
📄 [8/20] Scraping → https://innoacademy.engg.hku.hk/workshop/
📄 [9/20] Scraping → https://innoacademy.engg.hku.hk/sharing/
📄 [10/20] Scraping → https://innoacademy.engg.hku.hk/pitching/
📄 [11/20] Scraping → https://innoacademy.engg.hku.hk/sic/
📄 [12/20] Scraping → https://innoacademy.engg.hku.hk/studytour/
📄 [13/20] Scraping → https://innoacademy.engg.hku.hk/innoshow/
📄 [14/20] Scraping → https://innoacademy.engg.hku.hk/robotarm2026/
📄 [15/20] Scraping → https://innoacademy.engg.hku.hk/sharing/projects/
📄 [16/20] Scraping → https://innoacademy.engg.hku.hk/fund

## 1.6. Saving the Collected Data

We save everything as `data.json` — a clean, machine-readable format.

This file can later be loaded into a vector database for Retrieval-Augmented Generation (RAG) or a chatbot.

In [9]:
from dotenv import load_dotenv

load_dotenv("../.env")
DATASET = os.getenv("DATASET") or "data.json"

# Save to data.json
with open(DATASET, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"💾 Saved to {DATASET}.")

💾 Saved to data.json.


## 1.7. Data Quality Check (Very Important!)

Now let’s inspect what we actually collected.

### Questions to ask:
- Is the text clean and readable?
- Are there any noisy elements (menus, footers, JavaScript leftovers)?
- Does every page have meaningful content?
- Are there any error messages?

**Pro tip**: Always look at the first few and a few random pages before using the data!

🔍 First, we check the structure of an object.

In [10]:
# Print first thing scraped
print(documents[0])

{'url': 'https://innowings.engg.hku.hk/', 'text': 'Innovation Wing\nSkip to content\nThe Tam Wing Fan Innovation Wing One\nprovides an open environment to foster\ninterdisciplinary innovations among\nundergraduate students\nand teachers in Engineering and Technology. The provision of state-of-the-art facilities in a collaborative space will enable curriculum innovations that emphasize hands-on and experiential learning activities. The Innovation Wing serves as a platform to engage the young generation to explore the world, create opportunities for them to learn about the needs of the underprivileged, and acquire practical hands-on experience in developing solutions with real-world impact.\nInnovation Wing One\nThe Tam Wing Fan Innovation Wing Two\naims to serve as an enabling platform for Engineering\nresearchers\nto interact and collaborate synergistically with researchers and professionals across various disciplines to tackle grand challenges and deliver research outputs with signifi

🔍 Now let's check the content scraped.<br>
1. Is the data quality good?<br>
2. Are there rubbish information (=noisy data)?<br>

In [ ]:
# Print the content of this thing
print(documents[0]["text"])

Innovation Wing
Skip to content
The Tam Wing Fan Innovation Wing One
provides an open environment to foster
interdisciplinary innovations among
undergraduate students
and teachers in Engineering and Technology. The provision of state-of-the-art facilities in a collaborative space will enable curriculum innovations that emphasize hands-on and experiential learning activities. The Innovation Wing serves as a platform to engage the young generation to explore the world, create opportunities for them to learn about the needs of the underprivileged, and acquire practical hands-on experience in developing solutions with real-world impact.
Innovation Wing One
The Tam Wing Fan Innovation Wing Two
aims to serve as an enabling platform for Engineering
researchers
to interact and collaborate synergistically with researchers and professionals across various disciplines to tackle grand challenges and deliver research outputs with significant impact to Hong Kong and global communities, and at the sa

## 1.8. Conclusion & Next Steps

**Congratulations!** 🎉  
You have successfully built a polite recursive web scraper and collected data about HKU Innovation Wing & Innovation Academy.

### What you learned
- How to set up a controlled crawler
- Domain restriction and politeness rules
- HTML cleaning with BeautifulSoup
- Saving data for AI applications

### Possible next steps
1. Improve text cleaning (remove non-context texts)
2. Use website metadata (eg. headings) to further enhance the dataset
3. Add data from other sources